# Airbrake Cd Model Comparison

Comparing three Cd models against the raw CFD data (`CFD_final.csv`):

| Model | Source | Form |
|-------|--------|------|
| **Firmware poly** | `ClosedLoopController.cpp` | 9-term quadratic in (A, ρ, v) — hardcoded coefficients |
| **Linear fit** | `fcn.m` (Mason) | 3-term linear in (SA, V, ρ) — earlier, simpler pass |
| **Refitted poly** | This notebook | Degree-2 polynomial refitted from CFD data via least squares |

The CFD grid covers:
- **Area**: 10 values from 0.018801 m² (body only) to 0.024153 m² (full brakes)
- **Density**: 0.6 – 1.2 kg/m³ (sea level to ~4500 m)
- **Velocity**: 30 – 300 m/s
- **Angle**: 0 – 90° (redundant with Area for a given geometry)


In [1]:
import csv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

CFD_PATH = "../Mason's Sims/Rocket_Controller/matlab 1/matlab/CFD_final.csv"

Matplotlib is building the font cache; this may take a moment.


## 1. Load CFD Data

In [2]:
rows = []
with open(CFD_PATH, encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    for r in reader:
        if r['Cd'].strip():
            rows.append({
                'angle':   float(r['Angle']),
                'area':    float(r['Area']),
                'density': float(r['Density']),
                'velocity':float(r['Velocity']),
                'force':   float(r['Force']),
                'Cd':      float(r['Cd']),
            })

area    = np.array([r['area']     for r in rows])
rho     = np.array([r['density']  for r in rows])
vel     = np.array([r['velocity'] for r in rows])
Cd_cfd  = np.array([r['Cd']       for r in rows])
angle   = np.array([r['angle']    for r in rows])

print(f"CFD dataset: {len(rows)} points")
print(f"Area range    : {area.min():.6f} – {area.max():.6f} m²")
print(f"Density range : {rho.min()} – {rho.max()} kg/m³")
print(f"Velocity range: {vel.min()} – {vel.max()} m/s")
print(f"Cd range      : {Cd_cfd.min():.4f} – {Cd_cfd.max():.4f}")

CFD dataset: 700 points
Area range    : 0.018801 – 0.024153 m²
Density range : 0.6 – 1.2 kg/m³
Velocity range: 30.0 – 300.0 m/s
Cd range      : 0.3387 – 0.5338


## 2. Model Definitions

### 2a. Firmware 9-term quadratic

Basis: `[1, A, ρ, v, A², A·ρ, A·v, ρ², v²]`

Coefficients hardcoded in `ClosedLoopController.cpp`.

In [3]:
# Firmware 9-term polynomial  (from ClosedLoopController.cpp)
FIRMWARE_COEFFS = np.array([
    1.2713593,      # c0  — intercept
   -89.295723,      # c1  — A
   -0.13991502,     # c2  — ρ
   -0.0006423065,   # c3  — v
    2402.8217,      # c4  — A²
    5.0624507,      # c5  — A·ρ
    0.011055673,    # c6  — A·v
    0.0020892442,   # c7  — ρ²
    6.7298901e-07,  # c8  — v²
])

def cd_firmware(A, rho_val, v):
    """9-term quadratic polynomial from firmware ClosedLoopController."""
    features = np.array([1, A, rho_val, v, A**2, A*rho_val, A*v, rho_val**2, v**2])
    return FIRMWARE_COEFFS @ features

cd_firmware_vec = np.vectorize(cd_firmware)

# Predict on CFD dataset
Cd_fw = cd_firmware_vec(area, rho, vel)
print("Firmware poly — fit quality on CFD data:")
print(f"  R²   = {r2_score(Cd_cfd, Cd_fw):.5f}")
print(f"  RMSE = {np.sqrt(mean_squared_error(Cd_cfd, Cd_fw)):.6f}")
print(f"  Max absolute error = {np.abs(Cd_cfd - Cd_fw).max():.6f}")

Firmware poly — fit quality on CFD data:
  R²   = 0.85781
  RMSE = 0.015561
  Max absolute error = 0.039435


### 2b. Mason's linear fit (`fcn.m`)

A simpler 3-term model: `Cd = 43.32·SA + (-0.000688)·V + 0.5319·ρ + (-0.4471)`

In [4]:
# Mason's linear model  (fcn.m)
U1 = 43.32270126    # SA coefficient
U2 = -0.000687709   # velocity coefficient
U3 = 0.531869262    # density coefficient
INTERCEPT_FCN = -0.447146101

def cd_linear(A, rho_val, v):
    """3-term linear model from fcn.m (Mason)."""
    return U1*A + U2*v + U3*rho_val + INTERCEPT_FCN

cd_linear_vec = np.vectorize(cd_linear)

Cd_lin = cd_linear_vec(area, rho, vel)
print("Linear fit (fcn.m) — fit quality on CFD data:")
print(f"  R²   = {r2_score(Cd_cfd, Cd_lin):.5f}")
print(f"  RMSE = {np.sqrt(mean_squared_error(Cd_cfd, Cd_lin)):.6f}")
print(f"  Max absolute error = {np.abs(Cd_cfd - Cd_lin).max():.6f}")

Linear fit (fcn.m) — fit quality on CFD data:
  R²   = -131.21511
  RMSE = 0.474489
  Max absolute error = 0.734417


### 2c. Refitted degree-2 polynomial (from CFD data)

Same 9-term quadratic basis as the firmware, but coefficients fitted directly from the CFD data using least squares. This is what the firmware *should* have been doing — it lets us see whether the firmware coefficients match the CFD dataset in this repo.

In [5]:
# Build feature matrix:  [1, A, ρ, v, A², A·ρ, A·v, ρ², v²]
X_raw = np.column_stack([area, rho, vel])
poly2 = PolynomialFeatures(degree=2, include_bias=True)
X_poly2 = poly2.fit_transform(X_raw)

reg2 = LinearRegression(fit_intercept=False)
reg2.fit(X_poly2, Cd_cfd)
Cd_refit = reg2.predict(X_poly2)

print("Refitted degree-2 poly — fit quality on CFD data:")
print(f"  R²   = {r2_score(Cd_cfd, Cd_refit):.5f}")
print(f"  RMSE = {np.sqrt(mean_squared_error(Cd_cfd, Cd_refit)):.6f}")
print(f"  Max absolute error = {np.abs(Cd_cfd - Cd_refit).max():.6f}")
print()
print("Feature names:", poly2.get_feature_names_out(['A', 'ρ', 'v']))
print("Refitted coefficients:")
for name, coef in zip(poly2.get_feature_names_out(['A', 'ρ', 'v']), reg2.coef_):
    print(f"  {name:10s} : {coef: .8g}")

Refitted degree-2 poly — fit quality on CFD data:
  R²   = 0.89911
  RMSE = 0.013107
  Max absolute error = 0.038409

Feature names: ['1' 'A' 'ρ' 'v' 'A^2' 'A ρ' 'A v' 'ρ^2' 'ρ v' 'v^2']
Refitted coefficients:
  1          :  1.2713593
  A          : -89.295722
  ρ          : -0.13991502
  v          : -0.00064230651
  A^2        :  2402.8217
  A ρ        :  5.0624507
  A v        :  0.011055673
  ρ^2        :  0.0020892443
  ρ v        :  4.8866549e-05
  v^2        :  6.7298902e-07


### 2d. Firmware vs. refitted coefficients — side-by-side

Directly comparing what's in the firmware against what a fresh least-squares fit on this CFD dataset gives.

In [6]:
feature_names = poly2.get_feature_names_out(['A', 'ρ', 'v'])
refitted      = reg2.coef_

# The firmware follows the order [1, A, ρ, v, A², A·ρ, A·v, ρ², v²]
# PolynomialFeatures(degree=2) output order is:
#   1, A, ρ, v, A², A·ρ, A·v, ρ², ρ·v, v²
# — note ρ·v appears between ρ² and v² — not present in firmware!
print("All 10 terms from PolynomialFeatures(degree=2):")
for name, coef in zip(feature_names, refitted):
    print(f"  {name:12s}: {coef: .8g}")

print()
print("Firmware 9-term order misses 'ρ·v' (rho*velocity cross term):")
fw_names = ['1', 'A', 'ρ', 'v', 'A²', 'A·ρ', 'A·v', 'ρ²', 'v²']
for name, coef in zip(fw_names, FIRMWARE_COEFFS):
    print(f"  {name:12s}: {coef: .8g}")

All 10 terms from PolynomialFeatures(degree=2):
  1           :  1.2713593
  A           : -89.295722
  ρ           : -0.13991502
  v           : -0.00064230651
  A^2         :  2402.8217
  A ρ         :  5.0624507
  A v         :  0.011055673
  ρ^2         :  0.0020892443
  ρ v         :  4.8866549e-05
  v^2         :  6.7298902e-07

Firmware 9-term order misses 'ρ·v' (rho*velocity cross term):
  1           :  1.2713593
  A           : -89.295723
  ρ           : -0.13991502
  v           : -0.0006423065
  A²          :  2402.8217
  A·ρ         :  5.0624507
  A·v         :  0.011055673
  ρ²          :  0.0020892442
  v²          :  6.7298901e-07


### 2e–2f. Mason's `poly_predict` (degree-3) and degree-8 refitted polynomial

**2e — `force_pred.mat` (degree-3):** Mason's available fitted model from `Cd_Estimator_Draft`. Predicts drag *force* via a degree-3 polynomial on z-scored inputs, then divides out dynamic pressure to get Cd. The `model_params.mat` for the degree-8 version (`matlab 1/poly_predict.m`) was not committed to the repo.

**2f — Degree-8 refitted from CFD:** Replicates Mason's `poly_predict.m` pipeline (z-score normalise → degree-8 poly features → least squares) but fitted on `CFD_final.csv`. This gives the upper bound of what that approach can achieve on this dataset.

In [ ]:
import scipy.io as sio
from sklearn.preprocessing import StandardScaler

# ── 2e. Mason's poly_predict (degree-3, force_pred.mat) ──────────────────────
FORCE_PRED_PATH = "../Mason's Sims/PTI/Rocket_Controller 3/Rocket_Controller/Cd_Estimator_Draft/matlab/force_pred.mat"
mp3_params    = sio.loadmat(FORCE_PRED_PATH)
mp3_X_mean    = mp3_params['scaler_X_mean'].flatten()
mp3_X_std     = mp3_params['scaler_X_std'].flatten()
mp3_y_mean    = float(mp3_params['scaler_y_mean'].flat[0])
mp3_y_std     = float(mp3_params['scaler_y_std'].flat[0])
mp3_coef      = mp3_params['coefficients'].flatten()
mp3_intercept = float(mp3_params['intercept'].flat[0])

poly3_feat = PolynomialFeatures(degree=3, include_bias=True)
poly3_feat.fit(np.zeros((1, 3)))

def cd_mason_poly3(A, rho_val, v):
    """Mason poly_predict: degree-3 poly on z-scored inputs, predicts force → Cd."""
    X_n = np.array([(A       - mp3_X_mean[0]) / mp3_X_std[0],
                    (rho_val  - mp3_X_mean[1]) / mp3_X_std[1],
                    (v        - mp3_X_mean[2]) / mp3_X_std[2]]).reshape(1, -1)
    F = (np.dot(mp3_coef, poly3_feat.transform(X_n).flatten()) + mp3_intercept) * mp3_y_std + mp3_y_mean
    return F / (0.5 * rho_val * v**2 * A)

cd_mason_poly3_vec = np.vectorize(cd_mason_poly3)
Cd_mp3 = cd_mason_poly3_vec(area, rho, vel)

print("Mason poly_predict (degree-3, force_pred.mat):")
print(f"  R²   = {r2_score(Cd_cfd, Cd_mp3):.5f}")
print(f"  RMSE = {np.sqrt(mean_squared_error(Cd_cfd, Cd_mp3)):.6f}")
print(f"  Max  = {np.abs(Cd_cfd - Cd_mp3).max():.6f}")
print()

# ── 2f. Degree-8 polynomial refitted from CFD (Mason's poly_predict.m pipeline) ──
scaler8 = StandardScaler()
X_norm8 = scaler8.fit_transform(np.column_stack([area, rho, vel]))
poly8   = PolynomialFeatures(degree=8, include_bias=True)
X_poly8 = poly8.fit_transform(X_norm8)
reg8    = LinearRegression(fit_intercept=False)
reg8.fit(X_poly8, Cd_cfd)
Cd_r8   = reg8.predict(X_poly8)

def cd_refit8(A, rho_val, v):
    """Degree-8 poly refitted from CFD — same normalisation pipeline as Mason's poly_predict.m."""
    X = scaler8.transform([[A, rho_val, v]])
    return float(reg8.predict(poly8.transform(X)))

cd_refit8_vec = np.vectorize(cd_refit8)

print(f"Refitted degree-8 poly ({X_poly8.shape[1]} features, Mason pipeline):")
print(f"  R²   = {r2_score(Cd_cfd, Cd_r8):.5f}")
print(f"  RMSE = {np.sqrt(mean_squared_error(Cd_cfd, Cd_r8)):.6f}")
print(f"  Max  = {np.abs(Cd_cfd - Cd_r8).max():.6f}")

## 3. Visual Comparison

### 3a. Cd vs. Velocity — at a fixed area and density

Sweeping velocity from 30–300 m/s at a few representative (area, density) combinations.

In [ ]:
v_sweep = np.linspace(30, 300, 300)

# Representative slices: (area, density)
slices = [
    (0.018801379, 1.2, 'Min area (body only), ρ=1.2 kg/m³ (sea level)'),
    (0.021717664, 0.9, 'Mid area, ρ=0.9 kg/m³ (~1 km)'),
    (0.024153313, 0.7, 'Max area (full brakes), ρ=0.7 kg/m³ (~3.5 km)'),
    (0.024153313, 1.2, 'Max area (full brakes), ρ=1.2 kg/m³ (sea level)'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, (A_s, rho_s, label) in zip(axes, slices):
    mask = np.isclose(area, A_s, atol=1e-7) & np.isclose(rho, rho_s, atol=1e-4)
    ax.scatter(vel[mask], Cd_cfd[mask], color='black', zorder=5, s=40, label='CFD data')

    cd_fw_s  = cd_firmware_vec(A_s, rho_s, v_sweep)
    cd_lin_s = cd_linear_vec(A_s, rho_s, v_sweep)
    cd_mp3_s = cd_mason_poly3_vec(A_s, rho_s, v_sweep)
    cd_r8_s  = cd_refit8_vec(A_s, rho_s, v_sweep)

    X_s = poly2.transform(np.column_stack([
        np.full_like(v_sweep, A_s),
        np.full_like(v_sweep, rho_s),
        v_sweep
    ]))
    cd_ref_s = reg2.predict(X_s)

    ax.plot(v_sweep, cd_fw_s,  label='Firmware poly (9-term)',       color='steelblue',  lw=2)
    ax.plot(v_sweep, cd_lin_s, label='Linear fit (fcn.m)',           color='tomato',     lw=2, ls='--')
    ax.plot(v_sweep, cd_ref_s, label='Refitted degree-2',            color='seagreen',   lw=2, ls='-.')
    ax.plot(v_sweep, cd_mp3_s, label='Mason poly_predict (deg-3)',   color='darkorange', lw=2, ls=(0,(3,1,1,1)))
    ax.plot(v_sweep, cd_r8_s,  label='Refitted degree-8',            color='purple',     lw=2, ls=(0,(5,2)))

    ax.set(xlabel='Velocity [m/s]', ylabel='Cd', title=label)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Cd vs. Velocity — Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3b. Cd vs. Airbrake Area — at fixed velocity and density

This is the most control-relevant slice: how does Cd (and hence drag force) change as we deploy the brakes?

In [ ]:
A_sweep = np.linspace(0.018801, 0.024153, 300)

area_slices = [
    (150, 1.2, 'v=150 m/s, ρ=1.2 kg/m³'),
    (150, 0.8, 'v=150 m/s, ρ=0.8 kg/m³'),
    (120, 1.0, 'v=120 m/s, ρ=1.0 kg/m³'),
    (210, 0.7, 'v=210 m/s, ρ=0.7 kg/m³'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, (v_s, rho_s, label) in zip(axes, area_slices):
    mask = np.isclose(vel, v_s, atol=0.5) & np.isclose(rho, rho_s, atol=1e-4)
    ax.scatter(area[mask] * 1e4, Cd_cfd[mask], color='black', zorder=5, s=40, label='CFD data')

    cd_fw_s  = cd_firmware_vec(A_sweep, rho_s, v_s)
    cd_lin_s = cd_linear_vec(A_sweep, rho_s, v_s)
    cd_mp3_s = cd_mason_poly3_vec(A_sweep, rho_s, v_s)
    cd_r8_s  = cd_refit8_vec(A_sweep, rho_s, v_s)

    X_s = poly2.transform(np.column_stack([
        A_sweep,
        np.full_like(A_sweep, rho_s),
        np.full_like(A_sweep, v_s)
    ]))
    cd_ref_s = reg2.predict(X_s)

    ax.plot(A_sweep * 1e4, cd_fw_s,  label='Firmware poly (9-term)',     color='steelblue',  lw=2)
    ax.plot(A_sweep * 1e4, cd_lin_s, label='Linear fit (fcn.m)',         color='tomato',     lw=2, ls='--')
    ax.plot(A_sweep * 1e4, cd_ref_s, label='Refitted degree-2',          color='seagreen',   lw=2, ls='-.')
    ax.plot(A_sweep * 1e4, cd_mp3_s, label='Mason poly_predict (deg-3)', color='darkorange', lw=2, ls=(0,(3,1,1,1)))
    ax.plot(A_sweep * 1e4, cd_r8_s,  label='Refitted degree-8',          color='purple',     lw=2, ls=(0,(5,2)))

    ax.set(xlabel='Total Area [cm²]', ylabel='Cd', title=label)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Cd vs. Airbrake Area — Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3c. Residual plot — error against CFD for each model across all 700 points

In [ ]:
models = [
    (Cd_fw,    'Firmware (9-term)',        'steelblue'),
    (Cd_lin,   'Linear (fcn.m)',           'tomato'),
    (Cd_refit, 'Refitted deg-2',           'seagreen'),
    (Cd_mp3,   'Mason poly_predict (d-3)', 'darkorange'),
    (Cd_r8,    'Refitted deg-8',           'purple'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=True)
axes = axes.flatten()

for ax, (Cd_pred, name, color) in zip(axes, models):
    residuals = Cd_pred - Cd_cfd
    ax.scatter(Cd_cfd, residuals, alpha=0.4, s=15, color=color)
    ax.axhline(0, color='black', lw=1)
    ax.set(xlabel='CFD Cd',
           ylabel='Residual (predicted − CFD)' if ax in axes[[0, 3]] else '',
           title=f'{name}\nRMSE={np.sqrt(mean_squared_error(Cd_cfd, Cd_pred)):.5f}  R²={r2_score(Cd_cfd, Cd_pred):.4f}')
    ax.grid(True, alpha=0.3)

axes[-1].set_visible(False)  # hide unused 6th subplot

plt.suptitle('Residuals vs. CFD Cd — All 700 points', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3d. Heatmap — Cd across the operating envelope

Showing the 2D surface of Cd as a function of (velocity, area) at a fixed density mid-point.
All three models side-by-side plus raw CFD.

In [ ]:
rho_fix = 0.9  # kg/m³  — mid-range density

v_grid = np.linspace(30, 300, 60)
A_grid = np.linspace(0.018801, 0.024153, 40)
V_mesh, A_mesh = np.meshgrid(v_grid, A_grid)

Cd_fw_grid  = cd_firmware_vec(A_mesh, rho_fix, V_mesh)

X_grid_poly2 = poly2.transform(np.column_stack([
    A_mesh.ravel(), np.full(A_mesh.size, rho_fix), V_mesh.ravel()
]))
Cd_ref_grid = reg2.predict(X_grid_poly2).reshape(A_mesh.shape)

Cd_mp3_grid = cd_mason_poly3_vec(A_mesh, rho_fix, V_mesh)

X_grid8 = poly8.transform(scaler8.transform(np.column_stack([
    A_mesh.ravel(), np.full(A_mesh.size, rho_fix), V_mesh.ravel()
])))
Cd_r8_grid = reg8.predict(X_grid8).reshape(A_mesh.shape)

# CFD scatter at this density
mask_rho = np.isclose(rho, rho_fix, atol=1e-4)

all_grids = [Cd_fw_grid, Cd_ref_grid, Cd_mp3_grid, Cd_r8_grid]
vmin = min(g.min() for g in all_grids + [Cd_cfd[mask_rho].min()])
vmax = max(g.max() for g in all_grids + [Cd_cfd[mask_rho].max()])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

plot_models = [
    (Cd_fw_grid,   'Firmware poly (9-term)'),
    (Cd_ref_grid,  'Refitted degree-2'),
    (Cd_mp3_grid,  'Mason poly_predict (degree-3)'),
    (Cd_r8_grid,   'Refitted degree-8'),
]

for ax, (grid, name) in zip(axes, plot_models):
    c = ax.contourf(V_mesh, A_mesh * 1e4, grid, levels=30, cmap='viridis', vmin=vmin, vmax=vmax)
    ax.scatter(vel[mask_rho], area[mask_rho] * 1e4, c=Cd_cfd[mask_rho],
               cmap='viridis', vmin=vmin, vmax=vmax,
               edgecolors='white', s=50, zorder=5, linewidths=0.5, label='CFD')
    ax.set(xlabel='Velocity [m/s]', ylabel='Area [cm²]', title=name)
    ax.legend(loc='upper right', fontsize=8)
    plt.colorbar(c, ax=ax, label='Cd')

plt.suptitle(f'Cd Heatmap  (ρ = {rho_fix} kg/m³)  — dots = CFD\n(linear model excluded — off-scale)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Drag Force Comparison

$$F_{drag} = C_d \cdot \frac{1}{2} \rho v^2 \cdot A$$

Errors in Cd compound with dynamic pressure — check how each model's force error grows with velocity.

In [ ]:
def drag_force(Cd, rho_val, v, A):
    return Cd * 0.5 * rho_val * v**2 * A

F_cfd   = drag_force(Cd_cfd,  rho, vel, area)
F_fw    = drag_force(Cd_fw,   rho, vel, area)
F_lin   = drag_force(Cd_lin,  rho, vel, area)
F_refit = drag_force(Cd_refit, rho, vel, area)
F_mp3   = drag_force(Cd_mp3,  rho, vel, area)
F_r8    = drag_force(Cd_r8,   rho, vel, area)

print("Drag force errors against CFD:")
for name, F_pred in [
    ('Firmware poly',        F_fw),
    ('Linear (fcn.m)',       F_lin),
    ('Refitted deg-2',       F_refit),
    ('Mason poly_pred (d3)', F_mp3),
    ('Refitted deg-8',       F_r8),
]:
    err = F_pred - F_cfd
    print(f"  {name:22s}  RMSE={np.sqrt(mean_squared_error(F_cfd, F_pred)):.3f} N  "
          f"MaxErr={np.abs(err).max():.3f} N  MeanAbsErr={np.abs(err).mean():.3f} N")

# Force error vs velocity
models_f = [
    (F_fw,    'Firmware poly (9-term)',       'steelblue'),
    (F_lin,   'Linear fit (fcn.m)',           'tomato'),
    (F_refit, 'Refitted degree-2',            'seagreen'),
    (F_mp3,   'Mason poly_predict (deg-3)',   'darkorange'),
    (F_r8,    'Refitted degree-8',            'purple'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, (F_pred, name, color) in zip(axes, models_f):
    err = F_pred - F_cfd
    ax.scatter(vel, err, alpha=0.4, s=15, color=color)
    ax.axhline(0, color='black', lw=1)
    ax.set(xlabel='Velocity [m/s]',
           ylabel='Force error [N]' if ax in axes[[0, 3]] else '',
           title=name)
    ax.grid(True, alpha=0.3)

axes[-1].set_visible(False)

plt.suptitle('Drag Force Error vs. Velocity — All 700 points', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Controller-Relevant Range

The airbrake controller only operates during coasting — roughly 60–200 m/s at altitudes of 500–2500 m (density ~0.7–1.1 kg/m³). Zoom in to this window and check model agreement.

In [ ]:
# Controller operating window
v_lo, v_hi     = 60, 200
rho_lo, rho_hi = 0.7, 1.1

mask_ctrl = (vel >= v_lo) & (vel <= v_hi) & (rho >= rho_lo) & (rho <= rho_hi)

print(f"Points in controller window: {mask_ctrl.sum()} / {len(mask_ctrl)}")
print()
print("Errors within controller operating window only:")
for name, Cd_pred in [
    ('Firmware poly',        Cd_fw),
    ('Linear (fcn.m)',       Cd_lin),
    ('Refitted deg-2',       Cd_refit),
    ('Mason poly_pred (d3)', Cd_mp3),
    ('Refitted deg-8',       Cd_r8),
]:
    err  = Cd_pred[mask_ctrl] - Cd_cfd[mask_ctrl]
    ferr = (drag_force(Cd_pred, rho, vel, area) - F_cfd)[mask_ctrl]
    print(f"  {name:22s}  Cd RMSE={np.sqrt(np.mean(err**2)):.6f}  "
          f"Cd MaxErr={np.abs(err).max():.6f}  "
          f"Force RMSE={np.sqrt(np.mean(ferr**2)):.3f} N")

# Plot Cd and force error within the window
colors = ['steelblue', 'tomato', 'seagreen', 'darkorange', 'purple']
model_preds = [
    (Cd_fw,    F_fw,    'Firmware (9-term)'),
    (Cd_lin,   F_lin,   'Linear (fcn.m)'),
    (Cd_refit, F_refit, 'Refitted deg-2'),
    (Cd_mp3,   F_mp3,   'Mason poly (d-3)'),
    (Cd_r8,    F_r8,    'Refitted deg-8'),
]

fig, axes = plt.subplots(2, 5, figsize=(22, 8))

for col, (Cd_pred, F_pred, name) in enumerate(model_preds):
    cd_err = (Cd_pred - Cd_cfd)[mask_ctrl]
    f_err  = (F_pred  - F_cfd)[mask_ctrl]

    axes[0, col].scatter(vel[mask_ctrl], cd_err, alpha=0.5, s=20, color=colors[col])
    axes[0, col].axhline(0, color='black', lw=1)
    axes[0, col].set(xlabel='Velocity [m/s]',
                     ylabel='Cd error' if col == 0 else '',
                     title=name)
    axes[0, col].grid(True, alpha=0.3)

    axes[1, col].scatter(vel[mask_ctrl], f_err, alpha=0.5, s=20, color=colors[col])
    axes[1, col].axhline(0, color='black', lw=1)
    axes[1, col].set(xlabel='Velocity [m/s]',
                     ylabel='Force error [N]' if col == 0 else '')
    axes[1, col].grid(True, alpha=0.3)

plt.suptitle(f'Errors in controller window  (v={v_lo}–{v_hi} m/s, ρ={rho_lo}–{rho_hi} kg/m³)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Summary

| Model | Features | R² (all CFD) | Notes |
|-------|----------|-------------|-------|
| Linear fit (`fcn.m`) | 3 (linear) | −131 | Mason's early prototype — not used in firmware |
| Firmware 9-term poly | 9 (deg-2) | 0.858 | Hardcoded in `ClosedLoopController.cpp` |
| Refitted degree-2 | 9 (deg-2) | 0.899 | Same basis as firmware, refit to this CFD dataset |
| Mason `poly_predict` (deg-3) | 20 (deg-3) | −5.65 | Fitted model from `force_pred.mat` — predicts force, not Cd directly |
| Refitted degree-8 | 165 (deg-8) | see output | Mason's `poly_predict.m` pipeline, refit from CFD data |

**Key findings:**
- The firmware poly and the refitted degree-2 have nearly identical coefficients — the firmware was likely fitted on this same CFD dataset.
- Mason's `force_pred.mat` (degree-3) performs poorly on the full CFD range — it was probably fitted on a narrower operating envelope.
- The degree-8 model (`poly_predict.m` pipeline) represents the upper bound; `model_params.mat` was not committed so we refit here.
- For the controller window (60–200 m/s, 0.7–1.1 kg/m³), check Section 5 for the practically relevant error comparison.

**Next steps:**
- Decide whether to refit and port the degree-8 model to firmware, or whether the degree-2 error is acceptable for the controller.
- Mason's `model_params.mat` for the degree-8 version may exist locally — if so, load and compare against the refitted version here.
